# Notebook 2 — Create the Labels

## Goal

Create the `late_delivery` label using the actual delivery date and the estimated delivery date.

Only delivered orders can be labeled because we need an actual delivery date to decide whether the order was late or on time.

In [1]:
import os
import pandas as pd

## 1. Load the ML table

We read the artifact created by Notebook 1.

This keeps the notebooks connected in order without rebuilding the previous step.

In [2]:
ml_table1 = pd.read_csv("../artifacts/ml_table1.csv")

print("Shape:", ml_table1.shape)
ml_table1.head()

Shape: (99441, 26)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_product_weight,total_product_volume,number_of_product_categories,customer_seller_distance_km,customer_seller_same_state,number_of_payments,total_payment,average_installments,max_installments,main_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,500.0,1976.0,1.0,18.576109,True,3.0,38.71,1.0,1.0,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,400.0,4693.0,1.0,851.495057,False,1.0,141.46,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,420.0,9576.0,1.0,514.410611,False,1.0,179.12,3.0,3.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,450.0,6000.0,1.0,1822.226338,False,1.0,72.20,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,250.0,11475.0,1.0,29.676608,True,1.0,28.62,1.0,1.0,credit_card


## 2. Check the order status

We first check the order statuses to see how many orders were actually delivered.

In [3]:
ml_table1["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 3. Keep delivered orders

The target is based on the actual delivery date, so we only keep orders with `order_status = delivered`.

In [4]:
delivered_orders = ml_table1[
    ml_table1["order_status"] == "delivered"
].copy()

print("Delivered orders:", len(delivered_orders))

Delivered orders: 96478


## 4. Check the Required Dates

Both the actual delivery date and the estimated delivery date are required
to create the label.

We check for missing values before calculating the delivery delay.

In [5]:
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

delivered_orders[date_columns].isna().sum()

order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64

In [6]:
delivered_orders = delivered_orders.dropna(
    subset=date_columns
).copy()

print("Delivered orders after date check:", len(delivered_orders))

Delivered orders after date check: 96470


## 5. Convert the date columns

The delivery dates are stored as text in the CSV, so we convert them to datetime before comparing them.

In [7]:
for column in date_columns:
    delivered_orders[column] = pd.to_datetime(
        delivered_orders[column],
        errors="coerce"
    )

delivered_orders[date_columns].dtypes

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [8]:
delivered_orders[date_columns].isna().sum()

order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

## 6. Calculate the Delivery Delay

The delivery delay is calculated as:

`actual delivery date - estimated delivery date`

A positive value means that the order was delivered after the estimated date.
A zero or negative value means that the order was delivered on time or earlier.

In [9]:
delivered_orders["delivery_delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [10]:
delivered_orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,-5.543113
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,-11.461215
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,-31.410995
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,-6.281597
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,-8.528808


## 7. Create the Target Label

The target variable is `late_delivery`.

- `1` means the order was delivered after the estimated delivery date.
- `0` means the order was delivered on or before the estimated date.

In [11]:
delivered_orders["late_delivery"] = (
    delivered_orders["delivery_delay_days"] > 0
).astype(int)

## 8. Check the Created Label

Before using the target, we check several real orders and compare the
calculated delay with the generated label.

This verifies that the labeling rule was applied correctly.

In [12]:
delivered_orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days",
        "late_delivery"
    ]
].sample(10, random_state=42)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,late_delivery
9505,c6a73b421eb3e92ce86dbfbbd3530a8f,2018-03-13 17:13:17,2018-03-19,-5.282442,0
31124,b132124ca9d69faf63989e08a5851151,2018-06-04 19:26:52,2018-06-08,-3.189676,0
27722,7e25a1c58e68fa94300358caad65b944,2017-09-08 17:21:38,2017-09-25,-16.276644,0
74022,8418eb39cd68b52032566797b8f1bd11,2017-12-12 21:13:48,2017-12-20,-7.115417,0
36188,fe1ec86f91f3b5b6bc46fc3e4b8262cd,2017-12-13 01:16:52,2017-12-15,-1.946620,0
74560,c471a28c25283860362c67c0a1cd1694,2018-01-27 01:08:52,2018-02-22,-25.952176,0
27654,80e3ea9d65b89f8cbaa3d1f08f5dbd17,2018-06-19 21:50:42,2018-07-26,-36.089792,0
8692,a68c330c22c204ef5283a562bed75b97,2018-03-05 18:18:56,2018-03-13,-7.236852,0
50744,2728c4c5805dc2b5f0d4aee36cfde5d1,2017-06-19 18:16:09,2017-06-21,-1.238785,0
6939,d684202253f5f00621b0afd578646f3d,2017-10-10 20:21:31,2017-10-25,-14.151725,0


In [13]:
label_check = delivered_orders[
    [
        "delivery_delay_days",
        "late_delivery"
    ]
].copy()

label_check["expected_label"] = (
    label_check["delivery_delay_days"] > 0
).astype(int)

print(
    "Label mismatches:",
    (
        label_check["late_delivery"]
        != label_check["expected_label"]
    ).sum()
)

Label mismatches: 0


## 9. Check the Class Distribution

We now check how many orders are late and how many are on time.

This gives us an initial view of the target distribution and whether the
dataset has a class imbalance problem.

In [14]:
target_counts = (
    delivered_orders["late_delivery"]
    .value_counts()
    .sort_index()
)

target_counts

late_delivery
0    88644
1     7826
Name: count, dtype: int64

In [15]:
target_percentages = (
    delivered_orders["late_delivery"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

target_percentages.round(2)

late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64

## 10. Measure Class Imbalance

The positive class represents late deliveries.

We calculate the ratio between on-time and late orders to measure how
strong the class imbalance is.

The imbalance will be handled later during model training and evaluation.

In [16]:
on_time_count = target_counts.get(0, 0)
late_count = target_counts.get(1, 0)

imbalance_ratio = on_time_count / late_count

print("On-time orders:", on_time_count)
print("Late orders:", late_count)
print("On-time / Late ratio:", round(imbalance_ratio, 2))

On-time orders: 88644
Late orders: 7826
On-time / Late ratio: 11.33


## 11. Final Target Check

As a final check, we confirm that the target contains only two classes
and that no labels are missing.

In [17]:
print(
    "Missing labels:",
    delivered_orders["late_delivery"].isna().sum()
)

print(
    "Unique labels:",
    sorted(delivered_orders["late_delivery"].unique())
)

Missing labels: 0
Unique labels: [np.int64(0), np.int64(1)]


## 12. Save the Labeled Dataset

The labeled dataset is saved as an artifact for Notebook 3.

Notebook 3 will use this artifact to create the training, validation,
and test sets.

In [18]:
os.makedirs("../artifacts", exist_ok=True)

artifact_path = "../artifacts/labeled_orders1.csv"

delivered_orders.to_csv(
    artifact_path,
    index=False
)

print(f"Artifact saved to: {artifact_path}")

Artifact saved to: ../artifacts/labeled_orders1.csv


In [19]:
print("Artifact exists:", os.path.exists(artifact_path))
print("Saved rows:", len(delivered_orders))
print("Saved columns:", len(delivered_orders.columns))

Artifact exists: True
Saved rows: 96470
Saved columns: 28


In [20]:
print("=" * 70)
print("FINAL LABEL CHECK")
print("=" * 70)

print("\n1. Dataset shape")
print("Rows:", len(delivered_orders))
print("Columns:", len(delivered_orders.columns))

print("\n2. Order uniqueness")
print("Unique order IDs:", delivered_orders["order_id"].nunique())
print("Duplicate order IDs:", delivered_orders["order_id"].duplicated().sum())

print("\n3. Required dates")
print(
    delivered_orders[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].isna().sum()
)

print("\n4. Delivery delay")
print(
    delivered_orders["delivery_delay_days"].describe()
)

print("\n5. Label counts")
print(
    delivered_orders["late_delivery"]
    .value_counts()
    .sort_index()
)

print("\n6. Label percentages")
print(
    (
        delivered_orders["late_delivery"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

print("\n7. Label validation")
print(
    "Label mismatches:",
    (
        delivered_orders["late_delivery"]
        != (
            delivered_orders["delivery_delay_days"] > 0
        ).astype(int)
    ).sum()
)

print("\n8. Final target values")
print(
    sorted(delivered_orders["late_delivery"].unique())
)

print("\n9. Artifact")
print("Path:", artifact_path)
print("Exists:", os.path.exists(artifact_path))

FINAL LABEL CHECK

1. Dataset shape
Rows: 96470
Columns: 28

2. Order uniqueness
Unique order IDs: 96470
Duplicate order IDs: 0

3. Required dates
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

4. Delivery delay
count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64

5. Label counts
late_delivery
0    88644
1     7826
Name: count, dtype: int64

6. Label percentages
late_delivery
0    91.89
1     8.11
Name: proportion, dtype: float64

7. Label validation
Label mismatches: 0

8. Final target values
[np.int64(0), np.int64(1)]

9. Artifact
Path: ../artifacts/labeled_orders1.csv
Exists: True


## Final Summary

In this notebook, the target variable `late_delivery` was created.

An order is labeled as late when the actual customer delivery date is later than the estimated delivery date. Delivery-related information used to create the target is not used as a model feature later.

The labeled dataset was saved as an artifact for the next notebook.